# Account 360 and Customer 360 reporting - PySpark

Creates DataFrame-based historical reporting outputs from DWH facts, dimensions and bridges. It supports point-in-time analysis through `as_of_date`, preserves SCD-2 customer attributes, allocates joint-account activity, and prevents fact fanout by aggregating each fact grain before joining.

## Account 360

One row per account, combining current account ownership, as-of customer/KYC context, latest balance, transaction activity, late-arrival and reversal indicators, product usage, and risk flags. The balance aggregation is isolated from transaction aggregation to avoid multiplying account-day snapshots by transaction rows.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("environment", "dev", "Deployment Environment")
env = dbutils.widgets.get("environment")
catalog = f'{env}_banking'
reporting_schema = 'platinum'
dwh_schema='gold'

dbutils.widgets.text("as_of_date", "2024-12-31", "As-of business date")
dbutils.widgets.text("from_date", "2024-11-01", "Report start")
dbutils.widgets.text("to_date", "2024-12-31", "Report end")

as_of_date, from_date, to_date = [
    F.to_date(F.lit(dbutils.widgets.get(x))) for x in ["as_of_date", "from_date", "to_date"]
]
print(env, as_of_date, from_date, to_date)

In [0]:

def full_table_name(schema, name):
    return f"{catalog}.{schema}.{name}"

date = spark.table(full_table_name(dwh_schema, 'dim_date'))
account = spark.table(full_table_name(dwh_schema, 'dim_account'))
branch = spark.table(full_table_name(dwh_schema, 'dim_branch'))
customer = spark.table(full_table_name(dwh_schema, 'dim_customer'))
transactions = spark.table(full_table_name(dwh_schema, 'fact_transactions'))
balance = spark.table(full_table_name(dwh_schema, 'fact_account_balance'))
ownership = spark.table(full_table_name(dwh_schema, 'bridge_account_customer'))
product = spark.table(full_table_name(dwh_schema, 'dim_product'))
transaction_product = spark.table(full_table_name(dwh_schema, 'bridge_transaction_product'))

In [0]:
customer_asof = customer.filter(
    (F.col("StartDate") <= as_of_date) & (F.col("EndDate") > as_of_date)
)

account_asof = account.filter(
    (F.col("StartDate") <= as_of_date) & (F.col("EndDate") > as_of_date)
)

ownership_asof = ownership.filter(
    (F.col("StartDate") <= as_of_date) & (F.col("EndDate") > as_of_date)
)

period_txn = transactions.join(date.select("DateSK", "CalendarDate"), "DateSK").filter(
    (F.col("CalendarDate") >= from_date) & (F.col("CalendarDate") <= to_date)
)

posted_txn = period_txn.filter(F.col("TransactionStatus") == "POSTED")

In [0]:
accounts = account_asof.select(
    "AccountID","AccountSK",F.col("PrimaryCustomerID").alias("primary_customer_id"),F.col("BranchID").alias("branch_id"),F.col("AccountType").alias("account_type"),
    F.col("AccountStatus").alias("account_status"),"OpenDate","CloseDate",F.col("CurrencyCode").alias("currency_code"),
)
latest_account_balance = (
    balance.filter(F.col("BalanceDate") <= as_of_date)
    .withColumn("_rn", F.row_number().over(Window.partitionBy("AccountID").orderBy(F.col("BalanceDate").desc()))).filter("_rn=1").drop("_rn")
)
activity = period_txn.groupBy("AccountID").agg(
    F.countDistinct("TxnID").alias("TransactionCount"),
    F.sum(F.when(F.col("TransactionStatus") == "POSTED", F.col("TransactionAmount")).otherwise(F.lit(0))).alias("PostedNetAmount"),
    F.sum(F.when(F.col("TransactionStatus") == "REVERSED", 1).otherwise(0)).alias("ReversalCount"),
    F.max("CalendarDate").alias("LastTransactionDate"),
)

owner_context = ownership_asof.groupBy("AccountID").agg(
    F.countDistinct("CustomerID").alias("OwnerCount"),
    F.sum("OwnershipPercent").alias("OwnershipPercentTotal"),
)
account_360 = (
    accounts.join(latest_account_balance, "AccountID", "left")
    .join(activity, "AccountID", "left")
    .join(owner_context, "AccountID", "left")
    .join(branch, accounts.branch_id == branch.BranchID, "left")
    .join(customer_asof, accounts.primary_customer_id == customer_asof.CustomerID, "left")
    
    .select(accounts["*"], branch.BranchName, branch.Region, branch.State, customer_asof.CustomerSK, customer_asof.FullName, customer_asof.CustomerSegment, customer_asof.KYCStatus, latest_account_balance.LedgerBalanceAmount, latest_account_balance.AvailableBalanceAmount, latest_account_balance.OverdraftLimitAmount,
     activity.TransactionCount,activity.PostedNetAmount,activity.ReversalCount,activity.LastTransactionDate, owner_context.OwnerCount, owner_context.OwnershipPercentTotal,
    )
    .withColumn("IsOverdrawn", F.col("LedgerBalanceAmount") < 0)
    .withColumn("KYCReviewRequired", ~F.col("KYCStatus").isin("VERIFIED", "APPROVED"))
    .withColumn("IsJointAccount", F.col("OwnerCount") > 1)
    .withColumn("IsDormant",(F.col("LastTransactionDate").isNull()) | (F.datediff(as_of_date, F.col("LastTransactionDate")) > 90))
)
# account_360.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     f"{catalog}.{reporting_schema}.rpt_360_account"
# )
account_360.sort('LastTransactionDate').show()

## Branch 360

One row per branch as of the selected business date, combining account portfolio, customers, balances, transaction activity, overdraft exposure, joint-account penetration, dormancy and KYC review workload.

In [0]:
branch_360 = (
    account_360.groupBy('branch_id', "BranchName", "Region", "State")
    .agg(
        F.countDistinct("AccountID").alias("AccountCount"),
        F.countDistinct("CustomerSK").alias("CustomerCount"),
        F.sum("LedgerBalanceAmount").alias("LedgerBalance"),
        F.sum("AvailableBalanceAmount").alias("AvailableBalance"),
        F.sum("PostedNetAmount").alias("PostedNetActivity"),
        F.sum(F.when(F.col("IsOverdrawn"), 1).otherwise(0)).alias("OverdrawnAccountCount"),
        F.sum(F.when(F.col("IsJointAccount"), 1).otherwise(0)).alias("JointAccountCount"),
        F.sum(F.when(F.col("IsDormant"), 1).otherwise(0)).alias("DormantAccountCount"),
        F.sum(F.when(F.col("KYCReviewRequired"), 1).otherwise(0)).alias("KYCReviewAccountCount"),
        F.sum("ReversalCount").alias("ReversalCount"),
    )
    .withColumn("OverdraftExposure", F.greatest(-F.col("LedgerBalance"), F.lit(0)))
    .withColumn("JointAccountRate", F.when(F.col("AccountCount") == 0, F.lit(0)).otherwise(F.col("JointAccountCount") / F.col("AccountCount")))
    .withColumn("AsOfDate", as_of_date)
)
# branch_360.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     f"{catalog}.{reporting_schema}.rpt_360_branch"
# )

branch_360.show()

## Customer 360

One row per customer version valid on the as-of date. It includes owned-account counts, allocated balances and activity, product breadth, branch footprint, KYC/segment context, late-arrival exposure, and service/risk indicators. Allocation uses ownership percentage so joint accounts are not double-counted.

In [0]:
account_balances = (
    latest_account_balance.join(ownership_asof.select("AccountID", "CustomerID", "OwnershipPercent"), "AccountID",)
    .withColumn("AllocatedLedgerBalance",F.col("LedgerBalanceAmount") * F.col("OwnershipPercent") / 100)
    .withColumn("AllocatedAvailableBalance",F.col("AvailableBalanceAmount") * F.col("OwnershipPercent") / 100)
    .groupBy("CustomerID")
    .agg(
        F.countDistinct("AccountID").alias("OwnedAccountCount"),
        F.sum("AllocatedLedgerBalance").alias("AllocatedLedgerBalance"),
        F.sum("AllocatedAvailableBalance").alias("AllocatedAvailableBalance"),
        F.sum(F.when(F.col("LedgerBalanceAmount") < 0, -F.col("AllocatedLedgerBalance")).otherwise(0)
        ).alias("AllocatedOverdraftExposure"),
    )
)
ownership_bridge = ownership_asof.select("AccountID", "CustomerID", 'OwnershipPercent').distinct()

customer_activity = (
    posted_txn.join(ownership_bridge, "AccountID")
    .withColumn("TransactionAmount",F.col("TransactionAmount") * F.col("OwnershipPercent") / 100)
    .groupBy("CustomerID")
    .agg(
        F.countDistinct("TxnID").alias("TransactionCount"),
        F.sum("TransactionAmount").alias("PostedNetActivity"),
        F.max("CalendarDate").alias("LastActivityDate"),
        F.sum(F.when(F.col("TransactionStatus") == "REVERSED", 1).otherwise(0)).alias("ReversalCount")
    )
)

txn_accounts = period_txn.select("TxnID", "AccountID").distinct()
customer_products = (
    transaction_product.join(txn_accounts, "TxnID")
    .join(ownership_bridge, "AccountID")
    .withColumn("LineAmount",F.col("LineAmount") * F.col("OwnershipPercent") / 100)
    .groupBy("CustomerID")
    .agg(
        F.countDistinct("ProductID").alias("DistinctProductCount"),
        F.sum("LineAmount").alias("ProductLineAmount"),
    )
)
customer_branches = (
    account_asof.select("AccountID", F.col("BranchID").alias("BranchID"))
    .join(ownership_bridge, "AccountID")
    .groupBy("CustomerID")
    .agg(F.countDistinct("BranchID").alias("ServicingBranchCount"))
)

customer_360 = (
    customer_asof.join(account_balances, "CustomerID", "left")
    .join(customer_activity, "CustomerID", "left")
    .join(customer_products, "CustomerID", "left")
    .join(customer_branches, "CustomerID", "left")
    .fillna(0,
        subset=["OwnedAccountCount","AllocatedLedgerBalance","AllocatedAvailableBalance","AllocatedOverdraftExposure","TransactionCount","PostedNetActivity","ReversalCount","DistinctProductCount","ProductLineAmount","ServicingBranchCount"],
    )
    .withColumn("DaysSinceLastActivity", F.when(F.col("LastActivityDate").isNull(), F.lit(None)).otherwise(F.datediff(as_of_date, "LastActivityDate")))
    .withColumn("IsAtRisk",
        (F.col("AllocatedOverdraftExposure") > 0) | (F.col("KYCStatus").isin("PENDING", "EXPIRED")) | (F.col("DaysSinceLastActivity") > 90)
    )
    .withColumn("IsMultiProductCustomer", F.col("DistinctProductCount") > 1)
    .withColumn("AsOfDate", as_of_date)
    .select( "AsOfDate", "CustomerSK", "CustomerID", "FullName", "City", "State", "PostalCode", "CustomerSegment", "KYCStatus", "StartDate", "EndDate", "OwnedAccountCount", "AllocatedLedgerBalance", "AllocatedAvailableBalance", "AllocatedOverdraftExposure", "TransactionCount", "PostedNetActivity", "LastActivityDate", "DaysSinceLastActivity", "ReversalCount", "DistinctProductCount", "ProductLineAmount", "ServicingBranchCount", "IsAtRisk", "IsMultiProductCustomer",
    )
)
# customer_360.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
#     f"{catalog}.{reporting_schema}.rpt_360_customer"
# )
customer_360.show()

## Historical daily Customer 360 trend

This output supports BI trend charts and point-in-time snapshots. It is generated at customer-day grain from the daily balance fact and event fact, and can be materialized incrementally by processing date.

In [0]:
report_days = date.filter((F.col('CalendarDate') >= from_date) & (F.col('CalendarDate') <= to_date)).select('CalendarDate')

display(report_days.sort('CalendarDate'))

In [0]:
report_days = date.filter((F.col('CalendarDate') >= from_date) & (F.col('CalendarDate') <= to_date)).select('CalendarDate')

daily_balance = (
    balance.join(date.select('DateSK', 'CalendarDate'), 'DateSK')
    .join(
        ownership,
        (balance.AccountID == ownership.AccountID) &
        (F.col('CalendarDate') >= ownership.StartDate) &
        (F.col('CalendarDate') < ownership.EndDate),
        'inner'
    )
    .filter((F.col('CalendarDate') >= from_date) & (F.col('CalendarDate') <= to_date))
    .groupBy('CalendarDate', ownership.CustomerID)
    .agg(F.sum(F.col('LedgerBalanceAmount') * F.col('OwnershipPercent') / 100).alias('AllocatedLedgerBalance'),
         F.sum(F.col('AvailableBalanceAmount') * F.col('OwnershipPercent') / 100).alias('AllocatedAvailableBalance'),
    )
    .withColumnRenamed('CustomerID', 'BalanceCustomerID')
)

daily_activity = (
    period_txn.join(
        ownership,
        (period_txn.AccountID == ownership.AccountID) &
        (F.col('CalendarDate') >= ownership.StartDate) &
        (F.col('CalendarDate') < ownership.EndDate),
        'inner'
    )
    .groupBy('CalendarDate', ownership.CustomerID)
    .agg(
        # Owner-attributed count: a joint-account transaction appears once for each owner.
        F.countDistinct('TxnID').alias('TransactionCount'),
        # Weighted count: summing this across owners returns one transaction when
        # ownership percentages total 100%.
        F.sum(F.col('OwnershipPercent') / 100).alias('OwnershipWeightedTransactionCount'),
        F.sum(F.when(F.col('TransactionStatus') == 'POSTED', F.col('TransactionAmount') * F.col('OwnershipPercent') / 100,).otherwise(F.lit(0))).alias('PostedNetActivity'),
    )
    .withColumnRenamed('CustomerID', 'ActivityCustomerID')
)


# 2024-11-01
# 2024-11-02
# 2024-11-03
# 2024-11-04

# Customer attributes are also point-in-time: StartDate/EndDate select the
# dimension version, while CalendarDate remains the reporting grain.
customer_days = (customer.join(report_days, (F.col('CalendarDate') >= customer.StartDate) & (F.col('CalendarDate') < customer.EndDate), 'inner'
).select('CalendarDate', 'CustomerID', 'CustomerSegment', 'KYCStatus', 'State'))

customer_360_daily = (
    customer_days
    .join(daily_balance,
        (customer_days.CalendarDate == daily_balance.CalendarDate) &
        (customer_days.CustomerID == daily_balance.BalanceCustomerID),
        'left'
    )
    .join(
        daily_activity,
        (customer_days.CalendarDate == daily_activity.CalendarDate) &
        (customer_days.CustomerID == daily_activity.ActivityCustomerID),
        'left'
    )
    .select(customer_days.CalendarDate, customer_days.CustomerID, customer_days.CustomerSegment, customer_days.KYCStatus, customer_days.State, 'AllocatedLedgerBalance', 'AllocatedAvailableBalance', 'TransactionCount', 'PostedNetActivity',
    )
    .fillna(0, subset=['AllocatedLedgerBalance','AllocatedAvailableBalance','TransactionCount','PostedNetActivity',])
)

# customer_360_daily.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{reporting_schema}.rpt_360_customer_daily")
customer_360_daily.sort('CustomerID', 'CalendarDate').show()

In [0]:
display(customer_360_daily.sort('CustomerID', 'CalendarDate'))